In [5]:
import pickle
import numpy as np
from pathlib import Path 
import os 


with open("rsna_cta_valuations.pkl", "rb") as f:
    dict_data_loaded = pickle.load(f)

max_vals = np.array(dict_data_loaded["max_seg"])
count_1 = np.array(dict_data_loaded["count_1"])
count_2 = np.array(dict_data_loaded["count_2"])
valuations = np.array(dict_data_loaded["valuations"])
files = np.array(dict_data_loaded["filename"])

In [ ]:
path_crop = "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/crop_0.4_2_backup"

path_crop_2 = "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/crop_0.4_2"



# makedir 

os.makedirs(path_crop_2, exist_ok=True)

In [13]:
def get_closest_ref_value(val):
    ref_values = [-2048, -10048]
    ref_values = np.array(ref_values)
    diffs = np.abs(ref_values - val)
    idx = np.argmin(diffs)
    ref_value = ref_values[idx]
    if ref_value == -2048:
        return "case_one"
    if ref_value == -10048:
        return "case_two"

In [ ]:


i =0 
import shutil 
import SimpleITK as sitk 
for maxv, val, fil in zip(max_vals, valuations, files):
    i+=1
    if count_1[i-1] < 10000 or count_2[i-1] < 10000:
        print(str(i).rjust(6), "  ", maxv, str(int(np.mean(val))).rjust(8), str(count_1[i-1]).rjust(10), str(count_2[i-1]).rjust(10), " "*5, str(fil).ljust(60))
        # copy files
        src_file = Path(path_crop) / fil
        dst_file = Path(path_crop_2) / fil
        header = sitk.ReadImage(str(src_file))
        array = sitk.GetArrayFromImage(header)
        closest_value = get_closest_ref_value(np.mean(val))
        if closest_value == "case_one":
            array_fixed = array + 1024
        elif closest_value == "case_two":
            array_fixed = array/10 + 1024 
        

        image_fixed = sitk.GetImageFromArray(array_fixed)
        image_fixed.CopyInformation(header)
        sitk.WriteImage(image_fixed, str(dst_file))
        
        # fix intensities
        
        



In [1]:
import os
import SimpleITK as sitk

path_crop = "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/crop_0.4_3_backup"

path_crop_2 = "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/crop_0.4_3"



# makedir 

os.makedirs(path_crop_2, exist_ok=True)

In [2]:

exceptions = ["1.2.826.0.1.3680043.8.498.73287022341191968086339312998336143286.nii.gz"]
from pathlib import Path 
path_crop = Path(path_crop)
for fil in  path_crop.iterdir():
    if fil.name in exceptions:
        continue 
    header = sitk.ReadImage(str(fil))
    array = sitk.GetArrayFromImage(header)
    if fil.name == "1.2.826.0.1.3680043.8.498.70809927738286754136467574856093933410.nii.gz":
        array_fixed = array/10 + 1024
    elif fil.name == "1.2.826.0.1.3680043.8.498.73363802537202567743454640766773851605.nii.gz":
        array_fixed = array + 1024
    image_fixed = sitk.GetImageFromArray(array_fixed)
    image_fixed.CopyInformation(header)
    dst_file = Path(path_crop_2) / fil.name
    sitk.WriteImage(image_fixed, str(dst_file))
